# Random Forest 动态因子选股策略

## 论文参考

- **标题**: A Sustainable Quantitative Stock Selection Strategy Based on Dynamic Factor Adjustment
- **作者**: Yi Fu
- **年份**: 2020
- **DOI**: 10.3390/su12103978

### 摘要

本文提出了一种基于动态因子调整的可持续量化选股策略，使用随机森林 (Random Forest)
分类模型将股票分为上涨/下跌两类，通过滚动窗口动态调整因子权重和模型参数。
策略在 A 股市场回测中取得了 **年化 57% 的收益率** 和 **2.21 的夏普比率**，
显著优于传统多因子线性模型。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### Random Forest (随机森林)

随机森林是 Bagging 与随机子空间方法的结合：

**1. Bootstrap Aggregating (Bagging)**

从训练集 $D = \{(x_i, y_i)\}_{i=1}^n$ 中有放回抽取 $B$ 个子集，每个子集训练一棵树：
$$\hat{f}_{\text{RF}}(x) = \frac{1}{B} \sum_{b=1}^{B} T_b(x)$$

对分类问题采用多数投票：
$$\hat{y} = \arg\max_c \sum_{b=1}^{B} \mathbb{1}[T_b(x) = c]$$

**2. 随机特征子空间**

每棵树在每个分裂节点随机选择 $m = \lfloor\sqrt{p}\rfloor$ 个特征 (分类任务)，
降低树之间的相关性。

**3. OOB (Out-of-Bag) 误差估计**

每棵树约有 36.8% 的样本未被抽到 (OOB 样本)，可作为验证集：
$$\text{OOB Error} = \frac{1}{n} \sum_{i=1}^{n} \mathbb{1}\left[y_i \neq \hat{f}_{\text{OOB}}(x_i)\right]$$

**4. 特征重要性 (Gini Impurity Decrease)**

$$\text{Importance}(X_j) = \frac{1}{B}\sum_{b=1}^{B} \sum_{t \in T_b} \Delta \text{Gini}(t, X_j)$$

其中 $\Delta \text{Gini}(t, X_j) = \text{Gini}(t) - \frac{n_L}{n_t}\text{Gini}(t_L) - \frac{n_R}{n_t}\text{Gini}(t_R)$

**5. 选股逻辑**

- 将股票按未来 20 日收益率分为上涨 (Top 40%) 和下跌 (Bottom 40%) 两组
- 训练 Random Forest 分类器预测上涨概率
- 每月选择预测上涨概率最高的 Top-10 股票等权持有

In [ ]:
# ============================================================
# Cell 3: 动画展示 - Bagging: 多棵树并行生长与投票
# ============================================================
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_moons

np.random.seed(42)
X_demo, y_demo = make_moons(n_samples=200, noise=0.3, random_state=42)

xx, yy = np.meshgrid(np.linspace(-2, 3, 80), np.linspace(-1.5, 2, 80))
grid = np.column_stack([xx.ravel(), yy.ravel()])

n_trees_list = [1, 3, 5, 10, 20, 50, 100]
frames = []

for n_trees in n_trees_list:
    # 手动 bagging
    predictions = np.zeros((len(grid), 2))
    oob_correct = 0
    oob_total = 0

    for t in range(n_trees):
        idx = np.random.choice(len(X_demo), len(X_demo), replace=True)
        oob_idx = list(set(range(len(X_demo))) - set(idx))

        max_features = int(np.sqrt(X_demo.shape[1]))
        tree = DecisionTreeClassifier(max_depth=5, max_features=max_features,
                                      random_state=t)
        tree.fit(X_demo[idx], y_demo[idx])
        predictions += tree.predict_proba(grid)

        if oob_idx:
            oob_pred = tree.predict(X_demo[oob_idx])
            oob_correct += (oob_pred == y_demo[oob_idx]).sum()
            oob_total += len(oob_idx)

    avg_prob = predictions[:, 1] / n_trees
    oob_acc = oob_correct / oob_total if oob_total > 0 else 0
    ensemble_pred = (avg_prob > 0.5).astype(int)
    train_acc = (ensemble_pred[: len(X_demo)] == y_demo).mean() if n_trees > 0 else 0

    frames.append(go.Frame(
        data=[
            go.Heatmap(x=np.linspace(-2, 3, 80), y=np.linspace(-1.5, 2, 80),
                       z=avg_prob.reshape(xx.shape), colorscale='RdBu_r',
                       opacity=0.5, showscale=True,
                       colorbar=dict(title='P(class=1)')),
            go.Scatter(x=X_demo[y_demo == 0, 0], y=X_demo[y_demo == 0, 1],
                       mode='markers', name='Class 0',
                       marker=dict(color='#2196F3', size=5, line=dict(width=0.5, color='black'))),
            go.Scatter(x=X_demo[y_demo == 1, 0], y=X_demo[y_demo == 1, 1],
                       mode='markers', name='Class 1',
                       marker=dict(color='#F44336', size=5, line=dict(width=0.5, color='black'))),
        ],
        name=f'{n_trees} trees (OOB={oob_acc:.2%})'
    ))

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text='Random Forest: Bagging 集成 - 树数量对决策边界的影响'),
        xaxis_title='Feature 1', yaxis_title='Feature 2',
        height=550, width=700,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='▶ 播放', method='animate',
                 args=[None, {'frame': {'duration': 800}, 'transition': {'duration': 400}}]),
            dict(label='⏸ 暂停', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
            active=0,
        )],
    )
)
fig.show()

In [ ]:
# ============================================================
# Cell 4: 环境设置与导入
# ============================================================
import sys
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from datetime import datetime

warnings.filterwarnings('ignore')
np.random.seed(42)

sys.path.insert(0, '..')
from shared.data_fetcher import get_stock_daily, get_index_daily, get_multiple_stocks_daily
from shared.backtest_engine import MultiStockBacktester
from shared.factors import (
    momentum, volatility, rsi, macd, bollinger_bands, atr,
    turnover_factor, volume_price_corr, high_low_range, price_to_ma,
    winsorize, standardize
)
from shared.visualization import (
    plot_equity_curve, plot_drawdown, plot_metrics_table,
    plot_factor_importance, plot_returns_distribution
)
from shared.constants import DEFAULT_START, DEFAULT_END, CSI300_CODE

print('所有模块导入成功')
print(f'回测区间: {DEFAULT_START} - {DEFAULT_END}')

In [ ]:
# ============================================================
# Cell 5: 数据获取
# ============================================================

STOCK_POOL = [
    '600519', '601318', '600036', '000858', '601166',
    '600276', '601398', '600030', '000333', '002415',
    '600900', '601888', '000568', '002304', '601012',
    '600031', '603259', '600585', '000661', '601668',
]

try:
    stock_data = get_multiple_stocks_daily(STOCK_POOL, DEFAULT_START, DEFAULT_END)
    print(f'成功获取 {len(stock_data)} 只股票数据')
except Exception as e:
    print(f'数据获取异常: {e}, 使用模拟数据')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    stock_data = {}
    for sym in STOCK_POOL:
        np.random.seed(hash(sym) % 2**31)
        price = 50 + np.cumsum(np.random.randn(len(dates)) * 0.5)
        price = np.maximum(price, 5)
        stock_data[sym] = pd.DataFrame({
            'open': price * (1 + np.random.randn(len(dates)) * 0.005),
            'close': price,
            'high': price * (1 + np.abs(np.random.randn(len(dates)) * 0.01)),
            'low': price * (1 - np.abs(np.random.randn(len(dates)) * 0.01)),
            'volume': np.random.randint(100000, 5000000, len(dates)).astype(float),
            'turnover': np.random.uniform(0.3, 5.0, len(dates)),
        }, index=dates)

try:
    benchmark = get_index_daily(CSI300_CODE, DEFAULT_START, DEFAULT_END)
    benchmark_prices = benchmark['close']
    print(f'基准数据: {len(benchmark_prices)} 条')
except Exception as e:
    print(f'基准获取异常: {e}')
    dates = pd.bdate_range(DEFAULT_START, DEFAULT_END)
    benchmark_prices = pd.Series(5000 + np.cumsum(np.random.randn(len(dates)) * 10), index=dates)

In [ ]:
# ============================================================
# Cell 6: 因子工程 + 分类标签
# ============================================================

def build_features_rf(df):
    """为单只股票构建因子"""
    features = pd.DataFrame(index=df.index)

    # 动量
    mom = momentum(df['close'], periods=[5, 10, 20])
    features = pd.concat([features, mom], axis=1)

    # 波动率
    vol = volatility(df['close'], windows=[10, 20])
    features = pd.concat([features, vol], axis=1)

    # RSI, MACD, Bollinger
    features['rsi_14'] = rsi(df['close'], 14)
    macd_df = macd(df['close'])
    features['macd_hist'] = macd_df['histogram']
    bb = bollinger_bands(df['close'], 20)
    features['bb_pctb'] = bb['bb_pctb']
    features['bb_width'] = bb['bb_width']

    # 换手率
    if 'turnover' in df.columns:
        features['turnover'] = df['turnover']
        features['turnover_ma5'] = df['turnover'].rolling(5).mean()

    # 量价相关
    if 'volume' in df.columns:
        features['vp_corr_20'] = volume_price_corr(df['close'], df['volume'], 20)

    # 价格偏离
    ptm = price_to_ma(df['close'], windows=[5, 10, 20])
    features = pd.concat([features, ptm], axis=1)

    # ATR
    if all(c in df.columns for c in ['high', 'low']):
        features['atr_14'] = atr(df['high'], df['low'], df['close'], 14)
        features['hl_range'] = high_low_range(df['high'], df['low'], df['close'])

    return features


all_features = []
for sym, df in stock_data.items():
    if len(df) < 60:
        continue
    feats = build_features_rf(df)
    feats['fwd_return_20d'] = df['close'].pct_change(20).shift(-20)
    feats['symbol'] = sym
    all_features.append(feats)

panel = pd.concat(all_features).reset_index()
if 'index' in panel.columns:
    panel.rename(columns={'index': 'date'}, inplace=True)

FEATURE_COLS = [c for c in panel.columns if c not in ['date', 'symbol', 'fwd_return_20d', 'label']]

# 构建分类标签: Top 40% = 1 (上涨), Bottom 40% = 0 (下跌), 中间 20% 丢弃
panel['date'] = pd.to_datetime(panel['date'])
panel['year_month'] = panel['date'].dt.to_period('M')

def assign_label(group):
    ret = group['fwd_return_20d']
    q60 = ret.quantile(0.6)
    q40 = ret.quantile(0.4)
    group['label'] = np.nan
    group.loc[ret >= q60, 'label'] = 1
    group.loc[ret <= q40, 'label'] = 0
    return group

panel = panel.groupby('year_month', group_keys=False).apply(assign_label)

print(f'因子面板: {panel.shape[0]} 行 x {len(FEATURE_COLS)} 个因子')
print(f'标签分布:\n{panel["label"].value_counts(dropna=False)}')

In [ ]:
# ============================================================
# Cell 7: Random Forest 分类训练 (滚动窗口)
# ============================================================

TRAIN_MONTHS = 12
TOP_K = 10

months = sorted(panel['year_month'].unique())
print(f'数据覆盖 {len(months)} 个月')

selections = {}
all_importances = []
oob_scores = []

for i in range(TRAIN_MONTHS, len(months) - 1):
    train_months_range = months[i - TRAIN_MONTHS: i]
    test_month = months[i]

    train_data = panel[panel['year_month'].isin(train_months_range)].copy()
    test_data = panel[panel['year_month'] == test_month].copy()

    # 训练集: 去除中间标签
    train_data = train_data.dropna(subset=FEATURE_COLS + ['label'])
    test_data_clean = test_data.dropna(subset=FEATURE_COLS)

    if len(train_data) < 30 or len(test_data_clean) < 5:
        continue

    X_train = train_data[FEATURE_COLS].values
    y_train = train_data['label'].values.astype(int)
    X_test = test_data_clean[FEATURE_COLS].values

    # 训练 Random Forest
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        max_features='sqrt',
        min_samples_leaf=10,
        oob_score=True,
        n_jobs=-1,
        random_state=42,
    )
    rf.fit(X_train, y_train)

    oob_scores.append(rf.oob_score_)

    # 预测上涨概率
    prob = rf.predict_proba(X_test)[:, 1]
    test_data_clean = test_data_clean.copy()
    test_data_clean['prob_up'] = prob

    # 取最后一天截面选股
    last_day = test_data_clean.groupby('symbol')['prob_up'].last()
    top_stocks = last_day.nlargest(TOP_K).index.tolist()

    rebal_date = test_data_clean['date'].max()
    selections[rebal_date] = top_stocks

    # 特征重要性 (impurity decrease)
    imp = dict(zip(FEATURE_COLS, rf.feature_importances_))
    all_importances.append(imp)

    if (i - TRAIN_MONTHS) % 6 == 0:
        print(f'  {test_month}: OOB={rf.oob_score_:.3f}, 选股 {top_stocks[:5]}...')

print(f'\n共生成 {len(selections)} 期选股结果')
print(f'平均 OOB 准确率: {np.mean(oob_scores):.4f}')

# 平均特征重要性
avg_importance = {}
for col in FEATURE_COLS:
    vals = [imp.get(col, 0) for imp in all_importances]
    avg_importance[col] = np.mean(vals)

print('\nTop-10 重要因子 (Impurity Decrease):')
for k, v in sorted(avg_importance.items(), key=lambda x: -x[1])[:10]:
    print(f'  {k}: {v:.4f}')

In [ ]:
# ============================================================
# Cell 8: 信号生成与回测
# ============================================================

all_prices = {sym: df['close'] for sym, df in stock_data.items()}

backtester = MultiStockBacktester(initial_capital=1_000_000, rebalance_freq='M')
result = backtester.run(
    all_prices=all_prices,
    selections=selections,
    benchmark_prices=benchmark_prices
)

print('=== Random Forest 选股策略回测结果 ===')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 绩效可视化
# ============================================================
import matplotlib.pyplot as plt

# 1. 收益曲线
benchmark_equity = None
if result.get('benchmark_returns') is not None:
    benchmark_equity = (1 + result['benchmark_returns']).cumprod() * result['equity_curve'].iloc[0]
plot_equity_curve(result['equity_curve'], benchmark_equity,
                  title='Random Forest 选股 - 累计收益')

# 2. 回撤图
plot_drawdown(result['equity_curve'], title='Random Forest 选股 - 回撤')

# 3. 因子重要性 (Impurity Decrease)
plot_factor_importance(avg_importance,
                       title='Random Forest 因子重要性 (Gini Impurity Decrease)',
                       top_n=15)

# 4. OOB 准确率变化
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(len(oob_scores)), oob_scores, color='#2196F3', linewidth=1.5, marker='o', markersize=3)
ax.axhline(y=np.mean(oob_scores), color='red', linestyle='--', alpha=0.7,
           label=f'平均 OOB = {np.mean(oob_scores):.3f}')
ax.set_title('Random Forest OOB 准确率 (逐月)', fontsize=14, fontweight='bold')
ax.set_xlabel('月份序号', fontsize=12)
ax.set_ylabel('OOB 准确率', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 5. 收益率分布
plot_returns_distribution(result['returns'], title='Random Forest 策略日收益率分布')

# 6. 绩效表
plot_metrics_table(result['metrics'], title='Random Forest 选股策略绩效指标')

## 结果讨论

### Random Forest 分类方法的特点

1. **分类 vs 回归**: 将选股问题转化为分类问题，避免对收益率绝对值的预测，
   只需要排序准确即可
2. **OOB 误差**: 无需额外验证集即可估计泛化误差，是 Random Forest 的独特优势
3. **鲁棒性**: Bagging 通过采样多样性降低过拟合风险，对噪声数据更鲁棒

### 因子重要性分析

- 基于 Gini Impurity Decrease 的重要性反映了因子在树分裂中的贡献
- 动量因子和波动率因子通常排名靠前
- 换手率因子也有较高重要性，反映市场关注度对收益的影响

### 与 GBDT 方法的比较

| 特性 | Random Forest | LightGBM/XGBoost |
|------|--------------|------------------|
| 集成方式 | Bagging (并行) | Boosting (串行) |
| 偏差-方差 | 低方差，中偏差 | 低偏差，中方差 |
| 过拟合风险 | 较低 | 较高 (需正则化) |
| 超参数敏感 | 较低 | 较高 |

### 改进方向

- 动态调整分类阈值 (不固定 40%/60% 分位数)
- 使用 Permutation Importance 替代 Gini Importance
- 引入 Extra-Trees (极端随机树) 进一步增加随机性
- 结合 Random Forest 概率与其他模型做模型融合